# 15. 불량유형별 일반화 성능비교

scratch, impact, dent, stain별 실제 추론 성능을 비교합니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

In [ ]:
run_dir = paths.runs_root / "baseline_segformer_b0"
sample_metrics, group_metrics, class_metrics = load_run_metrics(run_dir)

## 15-1. 불량 유형별 target Dice와 FNR

In [ ]:
defect_table = plot_metric_bar(
    group_metrics,
    grouping="defect_type",
    metric="target_dice_mean",
    title="불량 유형별 defect Dice",
    out_path=paths.runs_root / "15_defect_target_dice.png",
)
display(defect_table)

fnr_table = plot_metric_bar(
    group_metrics,
    grouping="defect_type",
    metric="target_fnr_mean",
    title="불량 유형별 false negative rate",
    out_path=paths.runs_root / "15_defect_fnr.png",
)
display(fnr_table)

## 15-2. Defect x Color heatmap

In [ ]:
pivot = plot_metric_heatmap(
    sample_metrics,
    row="defect_type",
    col="color_group",
    metric="target_dice",
    title="defect x color target Dice",
    out_path=paths.runs_root / "15_defect_color_heatmap.png",
)
display(pivot)

## 15-3. 불량 유형 효과 결론

In [ ]:
print(conclusion_from_group(group_metrics, "defect_type", "target_dice_mean"))
if {"scratch", "impact"}.issubset(set(sample_metrics["defect_type"])):
    result = compare_two_groups(sample_metrics, "defect_type", "scratch", "impact", "target_dice")
    display(pd.DataFrame([result]))
    print(
        "결론:",
        "scratch와 impact의 차이는 bootstrap CI 기준으로 관측됩니다." if result["reject_h0_ci_excludes_0"]
        else "scratch와 impact의 차이는 현재 결과만으로 확정하기 어렵습니다."
    )